In [1]:
import pandas as pd

In [2]:
df = pd.read_pickle(r"D:\2026_Summer\TradingApp\data\v3.0\all_symbol_min_full_main_close_k_1.pkl")

In [3]:
ao = df[df['underlying_symbol'] == 'AO'].copy()
print(f'{len(ao):,} bars | {ao.trading_date.min().date()} -> {ao.trading_date.max().date()} '
      f'| {ao.trading_date.nunique()} sessions | contracts: {sorted(ao.contract.unique())}')

92,237 bars | 2023-06-27 -> 2024-04-22 | 200 sessions | contracts: ['AO2311', 'AO2312', 'AO2401', 'AO2402', 'AO2403', 'AO2404', 'AO2405', 'AO2406']


In [4]:
ao_daily = ao.groupby('trading_date').agg(
    bars      = ('close', 'size'),
    first_bar = ('close', lambda s: s.index.min().time()),
    last_bar  = ('close', lambda s: s.index.max().time()),
    close     = ('close', 'last'),
    volume    = ('volume', 'sum'),
    turnover  = ('total_turnover', 'sum'),
    oi_end    = ('open_interest', 'last'),
    contract  = ('contract', 'last'),
)
ao_daily['turnover_yi'] = (ao_daily['turnover'] / 1e8).round(2)

with pd.option_context('display.max_rows', None):
    display(ao_daily.drop(columns='turnover'))

,bars,first_bar,last_bar,close,volume,oi_end,contract,turnover_yi
trading_date,,,,,,,,
2023-06-27,466,21:00:00,15:00:00,2788.0,166881.0,56895.0,AO2311,93.52
2023-06-28,466,21:00:00,15:00:00,2791.0,93064.0,70248.0,AO2311,51.91
2023-06-29,466,21:00:00,15:00:00,2799.0,76690.0,81331.0,AO2311,42.88
2023-06-30,466,21:00:00,15:00:00,2812.0,75550.0,80875.0,AO2311,42.43
2023-07-03,466,21:00:00,15:00:00,2826.0,54744.0,86524.0,AO2311,30.88
2023-07-04,466,21:00:00,15:00:00,2816.0,53993.0,91724.0,AO2311,30.50
2023-07-05,466,21:00:00,15:00:00,2825.0,34750.0,92545.0,AO2311,19.63
2023-07-06,466,21:00:00,15:00:00,2834.0,44365.0,98721.0,AO2311,25.12
2023-07-07,466,21:00:00,15:00:00,2828.0,40786.0,101645.0,AO2311,23.10


In [5]:
tail = ao_daily.tail(20)
print('--- last 20 sessions ---')
print(tail[['bars', 'first_bar', 'last_bar', 'close', 'volume', 'oi_end', 'contract']])

print(f'\nbars in final session : {ao_daily.bars.iloc[-1]}   (median session: {ao_daily.bars.median():.0f})')
print(f'OI on final day       : {ao_daily.oi_end.iloc[-1]:,.0f}   (median: {ao_daily.oi_end.median():,.0f})')
print(f'volume last 5 vs med  : {ao_daily.volume.tail(5).mean():,.0f}  vs  {ao_daily.volume.median():,.0f}')
print(f'final contract        : {ao_daily.contract.iloc[-1]}  '
      f'(expiry implied {ao_daily.contract.iloc[-1][2:]}, last session {ao_daily.index[-1].date()})')

--- last 20 sessions ---
              bars first_bar  last_bar   close    volume   oi_end contract
trading_date                                                              
2024-03-22     466  21:00:00  15:00:00  3335.0  109359.0  57812.0   AO2405
2024-03-25     466  21:00:00  15:00:00  3290.0   78663.0  51097.0   AO2405
2024-03-26     466  21:00:00  15:00:00  3281.0   63001.0  48347.0   AO2405
2024-03-27     466  21:00:00  15:00:00  3303.0   59174.0  47048.0   AO2405
2024-03-28     466  21:00:00  15:00:00  3287.0   61047.0  44512.0   AO2405
2024-03-29     466  21:00:00  15:00:00  3295.0   53636.0  44199.0   AO2405
2024-04-01     466  21:00:00  15:00:00  3352.0   61387.0  47061.0   AO2405
2024-04-02     466  21:00:00  15:00:00  3326.0   41226.0  43818.0   AO2405
2024-04-03     466  21:00:00  15:00:00  3318.0   43246.0  39908.0   AO2405
2024-04-08     225  09:01:00  15:00:00  3344.0   45582.0  37484.0   AO2405
2024-04-09     466  21:00:00  15:00:00  3339.0   49462.0  34462.0   AO2405


In [6]:
# ============================================================
# 4. Is 2024-04-22 an AO problem or a cohort problem?
#    If every symbol that ends there also STARTS after ~2022,
#    it's the vendor's pipeline, not the market.
# ============================================================
life = (df.groupby('underlying_symbol')['trading_date']
          .agg(first='min', last='max')
          .sort_values('last'))
life['cohort'] = life['first'].dt.year

print('--- symbols by last date ---')
print(life.groupby('last').agg(n=('first','size'),
                               symbols=('first', lambda s: ' '.join(sorted(s.index)))
                               if False else ('first','size')))
print()
print(life[life['last'] < '2026-06-01'].to_string())

--- symbols by last date ---
             n  symbols
last                   
2024-04-22   9        9
2025-12-31   1        1
2026-01-16   8        8
2026-07-29  61       61

                       first       last  cohort
underlying_symbol                              
SI                2022-12-28 2024-04-22    2022
SH                2023-09-21 2024-04-22    2023
IM                2022-07-28 2024-04-22    2022
EC                2023-08-24 2024-04-22    2023
LC                2023-07-27 2024-04-22    2023
TL                2023-04-27 2024-04-22    2023
BR                2023-08-03 2024-04-22    2023
PX                2023-09-21 2024-04-22    2023
AO                2023-06-27 2024-04-22    2023
LR                2014-07-14 2025-12-31    2014
RI                2010-01-04 2026-01-16    2010
PM                2010-01-04 2026-01-16    2010
ZC                2013-10-09 2026-01-16    2013
RS                2013-01-08 2026-01-16    2013
WH                2010-01-04 2026-01-16    2010
JR        